# You Are Bot: final TF-IDF pipeline

In [4]:
import gc
import json
import random
import re
import warnings

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 240)

SEED = 42
N_SPLITS = 5
EPS = 1e-6

random.seed(SEED)
np.random.seed(SEED)


## 1. Загрузка данных

Если Kaggle положил файлы в другую папку, поправь только `DATA_DIR`.


In [5]:
DATA_DIR = "/kaggle/input/competitions/you-are-bot-2"

TRAIN_JSON_PATH = f"{DATA_DIR}/train.json"
TEST_JSON_PATH = f"{DATA_DIR}/test.json"
YTRAIN_PATH = f"{DATA_DIR}/ytrain.csv"
YTEST_PATH = f"{DATA_DIR}/ytest.csv"
SAMPLE_SUBMISSION_PATH = f"{DATA_DIR}/sample_submission.csv"


def load_dialogs(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if "root" in data:
        data = data["root"]
    return data


train_dialogs = load_dialogs(TRAIN_JSON_PATH)
test_dialogs = load_dialogs(TEST_JSON_PATH)
ytrain = pd.read_csv(YTRAIN_PATH)
ytest = pd.read_csv(YTEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Paths:")
print("train_json:", TRAIN_JSON_PATH)
print("test_json: ", TEST_JSON_PATH)
print("ytrain:    ", YTRAIN_PATH)
print("ytest:     ", YTEST_PATH)
print("sub:       ", SAMPLE_SUBMISSION_PATH)

print("\nShapes:")
print("train_dialogs", len(train_dialogs))
print("test_dialogs ", len(test_dialogs))
print("ytrain", ytrain.shape)
print("ytest ", ytest.shape)
print("sample_submission", sample_submission.shape)


Paths:
train_json: /kaggle/input/competitions/you-are-bot-2/train.json
test_json:  /kaggle/input/competitions/you-are-bot-2/test.json
ytrain:     /kaggle/input/competitions/you-are-bot-2/ytrain.csv
ytest:      /kaggle/input/competitions/you-are-bot-2/ytest.csv
sub:        /kaggle/input/competitions/you-are-bot-2/sample_submission.csv

Shapes:
train_dialogs 1516
test_dialogs  379
ytrain (3032, 3)
ytest  (758, 3)
sample_submission (758, 2)


## 2. Быстрое EDA


In [6]:
display(ytrain.head())
display(ytest.head())
display(sample_submission.head())

print("ytrain columns:", ytrain.columns.tolist())
print("ytest columns: ", ytest.columns.tolist())
print("submission columns:", sample_submission.columns.tolist())

print("\nTarget balance:")
display(ytrain["is_bot"].value_counts().to_frame("count"))
display(ytrain["is_bot"].value_counts(normalize=True).to_frame("share"))

pair_targets = (
    ytrain.sort_values(["dialog_id", "participant_index"])
    .groupby("dialog_id")["is_bot"]
    .apply(lambda x: tuple(x.tolist()))
)
print("\nTarget pairs per dialog:")
display(pair_targets.value_counts().to_frame("count"))
display(pair_targets.value_counts(normalize=True).to_frame("share"))

first_dialog_id = next(iter(train_dialogs))
print("\nFirst train dialog_id:", first_dialog_id)
print("Messages in first dialog:", len(train_dialogs[first_dialog_id]))
display(pd.DataFrame(train_dialogs[first_dialog_id]).head(10))


,dialog_id,participant_index,is_bot
0,0012a424-ae56-414b-a982-04cf24c35229,0,1
1,0012a424-ae56-414b-a982-04cf24c35229,1,0
2,009a6b25-0289-4845-a392-4025fde96371,0,1
3,009a6b25-0289-4845-a392-4025fde96371,1,0
4,00ade168-91d4-4605-b99b-da1ecbbb51fd,0,0


,dialog_id,participant_index,ID
0,0253c2df-7cea-4456-85d1-35f776c4f671,0,0253c2df-7cea-4456-85d1-35f776c4f671_0
1,0253c2df-7cea-4456-85d1-35f776c4f671,1,0253c2df-7cea-4456-85d1-35f776c4f671_1
2,03641877-db32-43b1-b78a-fba5a4aafa2d,0,03641877-db32-43b1-b78a-fba5a4aafa2d_0
3,03641877-db32-43b1-b78a-fba5a4aafa2d,1,03641877-db32-43b1-b78a-fba5a4aafa2d_1
4,0396d8a8-6f1b-437b-860e-2837683cb555,0,0396d8a8-6f1b-437b-860e-2837683cb555_0


,ID,is_bot
0,0253c2df-7cea-4456-85d1-35f776c4f671_0,0.5
1,0253c2df-7cea-4456-85d1-35f776c4f671_1,0.5
2,03641877-db32-43b1-b78a-fba5a4aafa2d_0,0.5
3,03641877-db32-43b1-b78a-fba5a4aafa2d_1,0.5
4,0396d8a8-6f1b-437b-860e-2837683cb555_0,0.5


ytrain columns: ['dialog_id', 'participant_index', 'is_bot']
ytest columns:  ['dialog_id', 'participant_index', 'ID']
submission columns: ['ID', 'is_bot']

Target balance:


,count
is_bot,
0,1795
1,1237


...


Target pairs per dialog:


,count
is_bot,
"(1, 0)",752
"(0, 1)",485
"(0, 0)",279


,share
is_bot,
"(1, 0)",0.496042
"(0, 1)",0.319921
"(0, 0)",0.184037



First train dialog_id: b5b8e0ae-5516-4150-9004-00fe422d5e33
Messages in first dialog: 9


,text,message,participant_index
0,"how can i helpt you, men?",0,0
1,Никак,1,1
2,неа,2,0
3,лох,3,1
4,не тупи,4,0
5,Дурак,5,1
6,"ага, ага, еще что скажешь, ботяра",6,0
7,Ьотяра,7,1
8,ты сломался кста,8,0


## 3. Сборка обучающей таблицы

Одна строка = один участник диалога. Храним отдельно текст участника, текст собеседника и весь диалог с маркерами `TARGET/OTHER`.


In [7]:
ID_COL = "ID"
TARGET = "is_bot"


def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def make_id(dialog_id, participant_index):
    return f"{dialog_id}_{int(participant_index)}"


def build_dataset(labels_df, dialogs, has_target):
    rows = []
    for row in labels_df.itertuples(index=False):
        dialog_id = str(getattr(row, "dialog_id"))
        participant_index = int(getattr(row, "participant_index"))
        messages = dialogs[dialog_id]

        own_messages = []
        other_messages = []
        marked_dialog = []
        own_lengths = []
        other_lengths = []

        for message in messages:
            text = clean_text(message.get("text", ""))
            author = int(message.get("participant_index"))
            if author == participant_index:
                own_messages.append(text)
                own_lengths.append(len(text))
                marked_dialog.append("TARGET: " + text)
            else:
                other_messages.append(text)
                other_lengths.append(len(text))
                marked_dialog.append("OTHER: " + text)

        item = {
            "ID": getattr(row, "ID") if "ID" in labels_df.columns else make_id(dialog_id, participant_index),
            "dialog_id": dialog_id,
            "participant_index": participant_index,
            "own_text": " ".join(own_messages),
            "other_text": " ".join(other_messages),
            "dialog_text": " ".join(marked_dialog),
            "n_own_messages": len(own_messages),
            "n_other_messages": len(other_messages),
            "n_messages": len(messages),
            "own_total_chars": sum(own_lengths),
            "other_total_chars": sum(other_lengths),
            "own_mean_chars": float(np.mean(own_lengths)) if own_lengths else 0.0,
            "other_mean_chars": float(np.mean(other_lengths)) if other_lengths else 0.0,
        }
        item["own_other_text"] = " [OWN] " + item["own_text"] + " [OTHER] " + item["other_text"]
        item["text"] = item["own_other_text"] + " [DIALOG] " + item["dialog_text"]
        if has_target:
            item[TARGET] = int(getattr(row, TARGET))
        rows.append(item)

    return pd.DataFrame(rows)


train = build_dataset(ytrain, train_dialogs, has_target=True)
test = build_dataset(ytest, test_dialogs, has_target=False)

print(train.shape, test.shape)
print("Submission ID matches test ID order:", sample_submission["ID"].tolist() == test["ID"].tolist())
display(train.head(3))
display(test.head(3))


(3032, 16) (758, 15)
Submission ID matches test ID order: True


,ID,dialog_id,participant_index,own_text,other_text,dialog_text,n_own_messages,n_other_messages,n_messages,own_total_chars,other_total_chars,own_mean_chars,other_mean_chars,own_other_text,text,is_bot
0,0012a424-ae56-414b-a982-04cf24c35229_0,0012a424-ae56-414b-a982-04cf24c35229,0,"Экономическая ситуация в Зимбабве за последнее время претерпела значительные изменения, и хотя она остаётся сложной, есть некоторые позитивные тенденции. ### Исторический контекст: - В конце 😄 Вот видишь, даже простое «ух ты» может заве...",ух ты про себя?,"TARGET: Экономическая ситуация в Зимбабве за последнее время претерпела значительные изменения, и хотя она остаётся сложной, есть некоторые позитивные тенденции. ### Исторический контекст: - В конце OTHER: ух ты TARGET: 😄 Вот видишь, да...",3,2,5,512,14,170.666667,7.000000,"[OWN] Экономическая ситуация в Зимбабве за последнее время претерпела значительные изменения, и хотя она остаётся сложной, есть некоторые позитивные тенденции. ### Исторический контекст: - В конце 😄 Вот видишь, даже простое «ух ты» мож...","[OWN] Экономическая ситуация в Зимбабве за последнее время претерпела значительные изменения, и хотя она остаётся сложной, есть некоторые позитивные тенденции. ### Исторический контекст: - В конце 😄 Вот видишь, даже простое «ух ты» мож...",1
1,0012a424-ae56-414b-a982-04cf24c35229_1,0012a424-ae56-414b-a982-04cf24c35229,1,ух ты про себя?,"Экономическая ситуация в Зимбабве за последнее время претерпела значительные изменения, и хотя она остаётся сложной, есть некоторые позитивные тенденции. ### Исторический контекст: - В конце 😄 Вот видишь, даже простое «ух ты» может заве...","OTHER: Экономическая ситуация в Зимбабве за последнее время претерпела значительные изменения, и хотя она остаётся сложной, есть некоторые позитивные тенденции. ### Исторический контекст: - В конце TARGET: ух ты OTHER: 😄 Вот видишь, даж...",2,3,5,14,512,7.000000,170.666667,"[OWN] ух ты про себя? [OTHER] Экономическая ситуация в Зимбабве за последнее время претерпела значительные изменения, и хотя она остаётся сложной, есть некоторые позитивные тенденции. ### Исторический контекст: - В конце 😄 Вот видишь, ...","[OWN] ух ты про себя? [OTHER] Экономическая ситуация в Зимбабве за последнее время претерпела значительные изменения, и хотя она остаётся сложной, есть некоторые позитивные тенденции. ### Исторический контекст: - В конце 😄 Вот видишь, ...",0
2,009a6b25-0289-4845-a392-4025fde96371_0,009a6b25-0289-4845-a392-4025fde96371,0,answer answer answer answer answer answer answer,"Привет Грустно, что ты не человек Любишь котиков? Бэ бэ бэ answer answer","TARGET: answer OTHER: Привет TARGET: answer OTHER: Грустно, что ты не человек TARGET: answer OTHER: Любишь котиков? TARGET: answer OTHER: Бэ бэ бэ TARGET: answer OTHER: answer TARGET: answer OTHER: answer TARGET: answer",7,6,13,42,67,6.000000,11.166667,"[OWN] answer answer answer answer answer answer answer [OTHER] Привет Грустно, что ты не человек Любишь котиков? Бэ бэ бэ answer answer","[OWN] answer answer answer answer answer answer answer [OTHER] Привет Грустно, что ты не человек Любишь котиков? Бэ бэ бэ answer answer [DIALOG] TARGET: answer OTHER: Привет TARGET: answer OTHER: Грустно, что ты не человек TARGET: answ...",1


,ID,dialog_id,participant_index,own_text,other_text,dialog_text,n_own_messages,n_other_messages,n_messages,own_total_chars,other_total_chars,own_mean_chars,other_mean_chars,own_other_text,text
0,0253c2df-7cea-4456-85d1-35f776c4f671_0,0253c2df-7cea-4456-85d1-35f776c4f671,0,Алл Прием Сосал?,Дла Меирм Ласос,TARGET: Алл OTHER: Дла TARGET: Прием OTHER: Меирм TARGET: Сосал? OTHER: Ласос,3,3,6,14,13,4.666667,4.333333,[OWN] Алл Прием Сосал? [OTHER] Дла Меирм Ласос,[OWN] Алл Прием Сосал? [OTHER] Дла Меирм Ласос [DIALOG] TARGET: Алл OTHER: Дла TARGET: Прием OTHER: Меирм TARGET: Сосал? OTHER: Ласос
1,0253c2df-7cea-4456-85d1-35f776c4f671_1,0253c2df-7cea-4456-85d1-35f776c4f671,1,Дла Меирм Ласос,Алл Прием Сосал?,OTHER: Алл TARGET: Дла OTHER: Прием TARGET: Меирм OTHER: Сосал? TARGET: Ласос,3,3,6,13,14,4.333333,4.666667,[OWN] Дла Меирм Ласос [OTHER] Алл Прием Сосал?,[OWN] Дла Меирм Ласос [OTHER] Алл Прием Сосал? [DIALOG] OTHER: Алл TARGET: Дла OTHER: Прием TARGET: Меирм OTHER: Сосал? TARGET: Ласос
2,03641877-db32-43b1-b78a-fba5a4aafa2d_0,03641877-db32-43b1-b78a-fba5a4aafa2d,0,да фигня какая-то... че надо? да ладно тебе......... привет-привет! ну и вопросы......... давай лучше о чём-нибудь интересном поговорим?,"опа,привет бот","TARGET: да фигня какая-то... че надо? OTHER: опа,привет бот TARGET: да ладно тебе......... привет-привет! ну и вопросы......... давай лучше о чём-нибудь интересном поговорим?",2,1,3,135,14,67.500000,14.000000,"[OWN] да фигня какая-то... че надо? да ладно тебе......... привет-привет! ну и вопросы......... давай лучше о чём-нибудь интересном поговорим? [OTHER] опа,привет бот","[OWN] да фигня какая-то... че надо? да ладно тебе......... привет-привет! ну и вопросы......... давай лучше о чём-нибудь интересном поговорим? [OTHER] опа,привет бот [DIALOG] TARGET: да фигня какая-то... че надо? OTHER: опа,привет бот ..."


## 4. Общие переменные и фичи


In [8]:
y = train[TARGET].astype(int).values
groups = train["dialog_id"].values
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(train, y, groups))

TEXT_COLS = ["own_text", "own_other_text", "dialog_text", "text"]
for col in TEXT_COLS:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)

NUMERIC_COLS = [
    "n_own_messages", "n_other_messages", "n_messages", "participant_index",
    "own_total_chars", "other_total_chars", "own_mean_chars", "other_mean_chars",
]
base_train_stats = train[NUMERIC_COLS].astype(float).values
base_test_stats = test[NUMERIC_COLS].astype(float).values

MODEL_RESULTS = {}


def add_model_result(name, oof_pred, test_pred):
    oof_pred = np.clip(np.asarray(oof_pred, dtype=float), EPS, 1 - EPS)
    test_pred = np.clip(np.asarray(test_pred, dtype=float), EPS, 1 - EPS)
    score = log_loss(y, oof_pred)
    MODEL_RESULTS[name] = {"oof": oof_pred, "test": test_pred, "score": score}
    print(f"{name}: OOF LogLoss = {score:.6f}")


print("y shape:", y.shape, "positive rate:", y.mean())
print("fold sizes:", [(len(tr), len(va)) for tr, va in folds])


y shape: (3032,) positive rate: 0.40798153034300794
fold sizes: [(2426, 606), (2426, 606), (2426, 606), (2426, 606), (2424, 608)]


## 5. Простые числовые признаки текста


In [9]:
def make_text_stats(texts):
    s = pd.Series(texts).fillna("").astype(str)
    char_len = s.str.len().astype(float)
    words = s.str.findall(r"\S+")
    word_count = words.map(len).astype(float)
    unique_word_count = words.map(lambda xs: len(set(w.lower() for w in xs))).astype(float)
    unique_ratio = unique_word_count / np.maximum(word_count, 1)
    avg_word_len = char_len / np.maximum(word_count, 1)
    digit_count = s.str.count(r"\d").astype(float)
    question_count = s.str.count(r"\?").astype(float)
    exclaim_count = s.str.count(r"!").astype(float)
    comma_count = s.str.count(r",").astype(float)
    dot_count = s.str.count(r"\.").astype(float)
    latin_count = s.str.count(r"[A-Za-z]").astype(float)
    cyrillic_count = s.str.count(r"[А-Яа-яЁё]").astype(float)
    upper_count = s.str.count(r"[A-ZА-ЯЁ]").astype(float)

    features = np.vstack([
        char_len,
        word_count,
        unique_word_count,
        unique_ratio,
        avg_word_len,
        digit_count,
        question_count,
        exclaim_count,
        comma_count,
        dot_count,
        latin_count / np.maximum(char_len, 1),
        cyrillic_count / np.maximum(char_len, 1),
        upper_count / np.maximum(char_len, 1),
    ]).T
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)


print(make_text_stats(train["text"].head()).shape)
display(train[["ID"] + NUMERIC_COLS].head())


(5, 13)


,ID,n_own_messages,n_other_messages,n_messages,participant_index,own_total_chars,other_total_chars,own_mean_chars,other_mean_chars
0,0012a424-ae56-414b-a982-04cf24c35229_0,3,2,5,0,512,14,170.666667,7.000000
1,0012a424-ae56-414b-a982-04cf24c35229_1,2,3,5,1,14,512,7.000000,170.666667
2,009a6b25-0289-4845-a392-4025fde96371_0,7,6,13,0,42,67,6.000000,11.166667
3,009a6b25-0289-4845-a392-4025fde96371_1,6,7,13,1,67,42,11.166667,6.000000
4,00ade168-91d4-4605-b99b-da1ecbbb51fd_0,1,1,2,0,2,36,2.000000,36.000000


## 6. TF-IDF

Public LB показал, что `own_text` лучше общего контекста: реплики собеседника часто шумят. Поэтому основной набор моделей обучается на тексте конкретного участника.


In [10]:
def build_tfidf_features(text_train, text_valid, text_test, stats_train, stats_valid, stats_test, cfg):
    blocks_train = []
    blocks_valid = []
    blocks_test = []

    if cfg.get("use_word", True):
        word_vectorizer = TfidfVectorizer(
            analyzer="word",
            ngram_range=cfg.get("word_ngram", (1, 2)),
            min_df=cfg.get("word_min_df", 2),
            max_df=cfg.get("word_max_df", 0.95),
            max_features=cfg.get("word_max_features", 250_000),
            sublinear_tf=cfg.get("sublinear_tf", True),
            strip_accents="unicode",
            lowercase=True,
        )
        blocks_train.append(word_vectorizer.fit_transform(text_train))
        blocks_valid.append(word_vectorizer.transform(text_valid))
        blocks_test.append(word_vectorizer.transform(text_test))

    if cfg.get("use_char", True):
        char_vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=cfg.get("char_ngram", (3, 5)),
            min_df=cfg.get("char_min_df", 2),
            max_df=cfg.get("char_max_df", 0.98),
            max_features=cfg.get("char_max_features", 300_000),
            sublinear_tf=cfg.get("sublinear_tf", True),
            lowercase=True,
        )
        blocks_train.append(char_vectorizer.fit_transform(text_train))
        blocks_valid.append(char_vectorizer.transform(text_valid))
        blocks_test.append(char_vectorizer.transform(text_test))

    if cfg.get("use_stats", True):
        text_stats_train = make_text_stats(text_train)
        text_stats_valid = make_text_stats(text_valid)
        text_stats_test = make_text_stats(text_test)

        num_train = np.hstack([text_stats_train, stats_train])
        num_valid = np.hstack([text_stats_valid, stats_valid])
        num_test = np.hstack([text_stats_test, stats_test])

        scaler = StandardScaler(with_mean=False)
        blocks_train.append(csr_matrix(scaler.fit_transform(num_train)))
        blocks_valid.append(csr_matrix(scaler.transform(num_valid)))
        blocks_test.append(csr_matrix(scaler.transform(num_test)))

    return (
        hstack(blocks_train, format="csr"),
        hstack(blocks_valid, format="csr"),
        hstack(blocks_test, format="csr"),
    )


def run_tfidf_model(name, text_col, cfg):
    oof = np.zeros(len(train), dtype=float)
    test_pred = np.zeros(len(test), dtype=float)
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(folds, 1):
        print(f"\n{name} | fold {fold}/{N_SPLITS}")
        X_tr, X_va, X_te = build_tfidf_features(
            train.iloc[tr_idx][text_col],
            train.iloc[va_idx][text_col],
            test[text_col],
            base_train_stats[tr_idx],
            base_train_stats[va_idx],
            base_test_stats,
            cfg,
        )
        print("features:", X_tr.shape)
        model = LogisticRegression(
            C=cfg.get("C", 3.0),
            solver=cfg.get("solver", "saga"),
            penalty="l2",
            max_iter=cfg.get("max_iter", 1000),
            class_weight=cfg.get("class_weight", None),
            n_jobs=-1,
            random_state=SEED + fold,
        )
        model.fit(X_tr, y[tr_idx])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        test_pred += model.predict_proba(X_te)[:, 1] / N_SPLITS
        fold_scores.append(log_loss(y[va_idx], np.clip(oof[va_idx], EPS, 1 - EPS)))
        print("fold logloss:", fold_scores[-1])
        del X_tr, X_va, X_te, model
        gc.collect()

    print("fold scores:", fold_scores)
    add_model_result(name, oof, test_pred)
    return oof, test_pred


TFIDF_CONFIGS = [
    ("own_w12_c25_C2", "own_text", {"word_ngram": (1, 2), "char_ngram": (2, 5), "C": 2.0}),
    ("own_w12_c35_C3_big", "own_text", {"word_ngram": (1, 2), "char_ngram": (3, 5), "C": 3.0, "word_max_features": 250_000, "char_max_features": 300_000}),
    ("own_w13_c36_C2", "own_text", {"word_ngram": (1, 3), "char_ngram": (3, 6), "C": 2.0, "word_max_features": 300_000, "char_max_features": 350_000}),
    ("own_w12_c26_C15", "own_text", {"word_ngram": (1, 2), "char_ngram": (2, 6), "C": 1.5, "word_max_features": 220_000, "char_max_features": 350_000}),
    ("own_w11_c27_C1", "own_text", {"word_ngram": (1, 1), "char_ngram": (2, 7), "C": 1.0, "word_max_features": 160_000, "char_max_features": 400_000}),
    ("own_char26_C2", "own_text", {"use_word": False, "char_ngram": (2, 6), "C": 2.0, "char_max_features": 450_000}),
    ("own_char37_C15", "own_text", {"use_word": False, "char_ngram": (3, 7), "C": 1.5, "char_max_features": 450_000}),
    ("own_word13_C2", "own_text", {"use_char": False, "word_ngram": (1, 3), "C": 2.0, "word_max_features": 300_000}),
    ("own_w12_c25_balanced", "own_text", {"word_ngram": (1, 2), "char_ngram": (2, 5), "C": 1.5, "class_weight": "balanced"}),
    ("full_old_like", "text", {"word_ngram": (1, 2), "char_ngram": (3, 5), "C": 3.0, "word_max_features": 250_000, "char_max_features": 300_000}),
]

for name, text_col, cfg in TFIDF_CONFIGS:
    run_tfidf_model(name, text_col, cfg)

pd.DataFrame({name: [res["score"]] for name, res in MODEL_RESULTS.items()}, index=["OOF LogLoss"]).T.sort_values("OOF LogLoss")



own_w12_c25_C2 | fold 1/5
features: (2426, 57683)
fold logloss: 0.3234609137676519

own_w12_c25_C2 | fold 2/5
features: (2426, 34980)
fold logloss: 0.3555795877912443

own_w12_c25_C2 | fold 3/5
features: (2426, 56801)
fold logloss: 0.33852150594091984

own_w12_c25_C2 | fold 4/5
features: (2426, 56158)
fold logloss: 0.3526509055208224

own_w12_c25_C2 | fold 5/5
features: (2424, 56012)
fold logloss: 0.43565361033674305
fold scores: [0.3234609137676519, 0.3555795877912443, 0.33852150594091984, 0.3526509055208224, 0.43565361033674305]
own_w12_c25_C2: OOF LogLoss = 0.361222

own_w12_c35_C3_big | fold 1/5
features: (2426, 52244)
fold logloss: 0.32408166134677135

own_w12_c35_C3_big | fold 2/5
features: (2426, 32772)
fold logloss: 0.3544524337099656

own_w12_c35_C3_big | fold 3/5
features: (2426, 51501)
fold logloss: 0.3378988999122228

own_w12_c35_C3_big | fold 4/5
features: (2426, 50816)
fold logloss: 0.3516435441885512

own_w12_c35_C3_big | fold 5/5
features: (2424, 50665)
fold logloss: 0

,OOF LogLoss
own_w12_c35_C3_big,0.360653
own_w12_c25_C2,0.361222
own_w13_c36_C2,0.362207
own_w12_c26_C15,0.362564
own_w11_c27_C1,0.365019
own_w12_c25_balanced,0.367288
own_char26_C2,0.373507
own_char37_C15,0.376518
own_word13_C2,0.379432
full_old_like,0.386057


## 7. Финальный отбор и сабмиты

Сохраняем несколько вариантов. В наших сабмитах лучший public score был у `submission_top3_weighted.csv` (`0.317`).


In [11]:
def summarize_models():
    rows = []
    for name, res in MODEL_RESULTS.items():
        rows.append({"model": name, "oof_logloss": res["score"]})
    return pd.DataFrame(rows).sort_values("oof_logloss").reset_index(drop=True)


def optimize_blend(names, n_iter=15000):
    oof_matrix = np.column_stack([MODEL_RESULTS[name]["oof"] for name in names])
    test_matrix = np.column_stack([MODEL_RESULTS[name]["test"] for name in names])

    mean_oof = np.clip(oof_matrix.mean(axis=1), EPS, 1 - EPS)
    best_score = log_loss(y, mean_oof)
    best_weights = np.ones(len(names)) / len(names)

    rng = np.random.default_rng(SEED + len(names))
    for _ in range(n_iter):
        weights = rng.dirichlet(np.ones(len(names)))
        pred = np.clip(oof_matrix @ weights, EPS, 1 - EPS)
        score = log_loss(y, pred)
        if score < best_score:
            best_score = score
            best_weights = weights

    weighted_oof = np.clip(oof_matrix @ best_weights, EPS, 1 - EPS)
    weighted_test = np.clip(test_matrix @ best_weights, EPS, 1 - EPS)
    mean_test = np.clip(test_matrix.mean(axis=1), EPS, 1 - EPS)
    return {
        "mean_oof": mean_oof,
        "mean_test": mean_test,
        "weighted_oof": weighted_oof,
        "weighted_test": weighted_test,
        "weighted_score": best_score,
        "weights": best_weights,
    }


summary = summarize_models()
display(summary)

best_single_name = summary.loc[0, "model"]
best_single_oof = MODEL_RESULTS[best_single_name]["oof"]
best_single_test = MODEL_RESULTS[best_single_name]["test"]
print("best single:", best_single_name, log_loss(y, best_single_oof))

TOP3 = summary.head(3)["model"].tolist()
TOP5 = summary.head(min(5, len(summary)))["model"].tolist()
print("TOP3:", TOP3)
print("TOP5:", TOP5)

blend3 = optimize_blend(TOP3)
blend5 = optimize_blend(TOP5)

print("top3 mean OOF:", log_loss(y, blend3["mean_oof"]))
print("top3 weighted OOF:", blend3["weighted_score"])
print("top5 mean OOF:", log_loss(y, blend5["mean_oof"]))
print("top5 weighted OOF:", blend5["weighted_score"])

weights3 = pd.DataFrame({"model": TOP3, "weight": blend3["weights"], "score": [MODEL_RESULTS[n]["score"] for n in TOP3]})
weights5 = pd.DataFrame({"model": TOP5, "weight": blend5["weights"], "score": [MODEL_RESULTS[n]["score"] for n in TOP5]})
print("\nTop3 weights:")
display(weights3.sort_values("weight", ascending=False))
print("\nTop5 weights:")
display(weights5.sort_values("weight", ascending=False))

print("Best single TEST distribution:")
display(pd.Series(best_single_test).describe())
print("Top3 weighted TEST distribution:")
display(pd.Series(blend3["weighted_test"]).describe())


,model,oof_logloss
0,own_w12_c35_C3_big,0.360653
1,own_w12_c25_C2,0.361222
2,own_w13_c36_C2,0.362207
3,own_w12_c26_C15,0.362564
4,own_w11_c27_C1,0.365019
5,own_w12_c25_balanced,0.367288
6,own_char26_C2,0.373507
7,own_char37_C15,0.376518
8,own_word13_C2,0.379432
9,full_old_like,0.386057


best single: own_w12_c35_C3_big 0.3606528491403423
TOP3: ['own_w12_c35_C3_big', 'own_w12_c25_C2', 'own_w13_c36_C2']
TOP5: ['own_w12_c35_C3_big', 'own_w12_c25_C2', 'own_w13_c36_C2', 'own_w12_c26_C15', 'own_w11_c27_C1']
top3 mean OOF: 0.3612158619510092
top3 weighted OOF: 0.36064693181338553
top5 mean OOF: 0.36207007722153167
top5 weighted OOF: 0.36073823239011416

Top3 weights:


,model,weight,score
0,own_w12_c35_C3_big,0.953793,0.360653
1,own_w12_c25_C2,0.045303,0.361222
2,own_w13_c36_C2,0.000904,0.362207



Top5 weights:


,model,weight,score
0,own_w12_c35_C3_big,0.868611,0.360653
1,own_w12_c25_C2,0.081884,0.361222
2,own_w13_c36_C2,0.032378,0.362207
4,own_w11_c27_C1,0.016804,0.365019
3,own_w12_c26_C15,0.000323,0.362564


Best single TEST distribution:


count    758.000000
mean       0.422051
std        0.345421
min        0.004558
25%        0.086387
50%        0.319160
75%        0.760735
max        0.999999
dtype: float64

Top3 weighted TEST distribution:


count    758.000000
mean       0.422048
std        0.345313
min        0.004580
25%        0.086568
50%        0.319179
75%        0.760287
max        0.999999
dtype: float64

In [12]:
def save_submission(filename, pred):
    sub = sample_submission.copy()
    sub["is_bot"] = np.clip(pred, EPS, 1 - EPS)
    sub.to_csv(filename, index=False)
    print("Saved", filename, sub.shape)
    return sub


submission_best_single = save_submission("submission_best_single.csv", best_single_test)
submission_top3_mean = save_submission("submission_top3_mean.csv", blend3["mean_test"])
submission_top3_weighted = save_submission("submission_top3_weighted.csv", blend3["weighted_test"])
submission_top5_mean = save_submission("submission_top5_mean.csv", blend5["mean_test"])
submission_top5_weighted = save_submission("submission_top5_weighted.csv", blend5["weighted_test"])

# Kaggle default file: лучший public LB у нас был top3 weighted.
submission = save_submission("submission.csv", blend3["weighted_test"])
display(submission.head())


Saved submission_best_single.csv (758, 2)
Saved submission_top3_mean.csv (758, 2)
Saved submission_top3_weighted.csv (758, 2)
Saved submission_top5_mean.csv (758, 2)
Saved submission_top5_weighted.csv (758, 2)
Saved submission.csv (758, 2)


,ID,is_bot
0,0253c2df-7cea-4456-85d1-35f776c4f671_0,0.066952
1,0253c2df-7cea-4456-85d1-35f776c4f671_1,0.354017
2,03641877-db32-43b1-b78a-fba5a4aafa2d_0,0.975534
3,03641877-db32-43b1-b78a-fba5a4aafa2d_1,0.080130
4,0396d8a8-6f1b-437b-860e-2837683cb555_0,0.142359


## Итог

Лучше всего сработал простой подход: TF-IDF по собственным репликам участника (own_text) + LogisticRegression. Добавление контекста диалога, реплик собеседника и более сложных моделей в основном ухудшало качество.

Что пробовал:

Baseline TF-IDF full text: около 0.33 на public LB.
Own-text TF-IDF: улучшил результат до примерно 0.318.
Top-3 weighted ensemble TF-IDF моделей: лучший public score 0.317.
KNN / clustering features: оказались слабыми, качество сильно хуже TF-IDF.
Frozen BERT embeddings + LogisticRegression: сильно хуже, OOF около 0.55.
BERT fine-tuning: validation LogLoss был около 0.49+, тоже хуже TF-IDF.
Главное наблюдение: задача больше про стиль, шаблоны, повторы, мусорные фразы, пунктуацию и короткие реплики, чем про глубокую семантику. Поэтому символные и словесные TF-IDF признаки оказались сильнее BERT-подходов.

Финальное решение: ансамбль top-3 TF-IDF моделей по own_text, файл submission.csv, public score около 0.317.